#KPI 1: Monthly Consolidated Revenue with MoM Growth


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_monthly_revenue AS
WITH monthly_revenue AS (
  SELECT 
    year_month,
    year,
    month,
    SUM(gl_revenue) AS total_revenue,
    LAG(SUM(gl_revenue)) OVER (ORDER BY year_month) AS prev_month_revenue
  FROM mini_project.gold.datacube_kpi
  GROUP BY year_month, year, month
)
SELECT 
  year_month,
  year,
  month,
  ROUND(total_revenue, 2) AS total_revenue,
  ROUND(prev_month_revenue, 2) AS prev_month_revenue,
  ROUND(total_revenue - COALESCE(prev_month_revenue, 0), 2) AS revenue_change,
  CASE 
    WHEN prev_month_revenue IS NULL OR prev_month_revenue = 0 THEN NULL
    ELSE ROUND(((total_revenue - prev_month_revenue) / prev_month_revenue) * 100, 2)
  END AS mom_growth_pct,
  current_timestamp() AS last_updated
FROM monthly_revenue
ORDER BY year_month;



#KPI 2: Cost of Sales by Month and Company


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_cost_of_sales AS
SELECT 
  year_month,
  year,
  month,
  company_name,
  ROUND(SUM(gl_cogs), 2) AS cost_of_sales,
  SUM(gl_transaction_count) AS transaction_count,
  current_timestamp() AS last_updated
FROM mini_project.gold.datacube_kpi
GROUP BY year_month, year, month, company_name
ORDER BY year_month, company_name;



#KPI 3: Monthly Gross Profit Margin


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_gross_profit_margin AS
SELECT 
  year_month,
  year,
  month,
  company_id,
  company_name,
  ROUND(SUM(gl_revenue), 2) AS revenue,
  ROUND(SUM(gl_cogs), 2) AS cost_of_sales,
  ROUND(SUM(gross_profit), 2) AS gross_profit,
  CASE 
    WHEN SUM(gl_revenue) = 0 THEN 0
    ELSE ROUND((SUM(gross_profit) / SUM(gl_revenue)) * 100, 2)
  END AS gross_profit_margin_pct,
  current_timestamp() AS last_updated
FROM mini_project.gold.datacube_kpi
GROUP BY year_month, year, month, company_id, company_name
ORDER BY year_month, company_id;



#KPI 4: Operating Expense Breakdown by Category


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_operating_expenses AS
WITH expense_breakdown AS (
  SELECT 
    year_month,
    year,
    month,
    company_id,
    company_name,
    'Personnel Expense' AS expense_category,
    gl_personnel_expense AS expense_amount
  FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Occupancy Expense', gl_occupancy_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Technology Expense', gl_technology_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Marketing Expense', gl_marketing_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Travel & Entertainment', gl_travel_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Professional Services', gl_professional_services FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Communication Expense', gl_communication_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Financial Expense', gl_financial_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Depreciation & Amortization', gl_depreciation FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Other Operating Expense', gl_other_opex FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'R&D Expense', gl_rnd_expense FROM mini_project.gold.datacube_kpi
  UNION ALL
  SELECT year_month, year, month, company_id, company_name, 'Tax Expense', gl_tax_expense FROM mini_project.gold.datacube_kpi
)
SELECT 
  year_month,
  year,
  month,
  company_id,
  company_name,
  expense_category,
  ROUND(SUM(expense_amount), 2) AS expense_amount,
  current_timestamp() AS last_updated
FROM expense_breakdown
WHERE expense_amount > 0
GROUP BY year_month, year, month, company_id, company_name, expense_category
ORDER BY year_month, company_id, expense_amount DESC;



#KPI 5: Average Compensation by Position


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_avg_compensation_by_position AS
SELECT 
  position,
  company_id,
  company_name,
  COUNT(DISTINCT employee_id) AS employee_count,
  ROUND(AVG(gross_salary), 2) AS avg_base_salary,
  ROUND(AVG(bonus), 2) AS avg_bonus,
  ROUND(AVG(overtime_pay), 2) AS avg_overtime,
  ROUND(AVG(commission), 2) AS avg_commission,
  ROUND(AVG(allowances), 2) AS avg_allowances,
  ROUND(AVG(total_compensation), 2) AS avg_total_compensation,
  ROUND(MIN(total_compensation), 2) AS min_compensation,
  ROUND(MAX(total_compensation), 2) AS max_compensation,
  current_timestamp() AS last_updated
FROM mini_project.gold.fact_payroll
WHERE status IN ('Paid', 'Processing')
GROUP BY position, company_id, company_name
ORDER BY company_id, avg_total_compensation DESC;



#KPI 6: Net Profit by Month


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_net_profit AS
SELECT 
  year_month,
  year,
  month,
  company_id,
  company_name,
  ROUND(SUM(gl_revenue), 2) AS total_revenue,
  ROUND(SUM(gl_cogs), 2) AS total_cogs,
  ROUND(SUM(gross_profit), 2) AS gross_profit,
  ROUND(SUM(gl_total_expenses), 2) AS total_operating_expenses,
  ROUND(SUM(net_profit), 2) AS net_profit,
  CASE 
    WHEN SUM(gl_revenue) = 0 THEN 0
    ELSE ROUND((SUM(net_profit) / SUM(gl_revenue)) * 100, 2)
  END AS net_profit_margin_pct,
  current_timestamp() AS last_updated
FROM mini_project.gold.datacube_kpi
GROUP BY year_month, year, month, company_id, company_name
ORDER BY year_month, company_id;



#KPI 7: Overtime and Bonus Analysis by Department


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_overtime_bonus_analysis AS
SELECT 
  year_month,
  year,
  month,
  company_id,
  company_name,
  department_id,
  department_name,
  employee_count,
  ROUND(payroll_gross_salary, 2) AS total_base_salary,
  ROUND(payroll_overtime, 2) AS total_overtime,
  ROUND(payroll_bonus, 2) AS total_bonus,
  ROUND(payroll_commission, 2) AS total_commission,
  ROUND(avg_overtime_ratio, 2) AS avg_overtime_to_base_ratio,
  CASE 
    WHEN payroll_gross_salary = 0 THEN 0
    ELSE ROUND((payroll_overtime / payroll_gross_salary) * 100, 2)
  END AS overtime_pct_of_base,
  CASE 
    WHEN payroll_gross_salary = 0 THEN 0
    ELSE ROUND((payroll_bonus / payroll_gross_salary) * 100, 2)
  END AS bonus_pct_of_base,
  current_timestamp() AS last_updated
FROM mini_project.gold.datacube_kpi
WHERE employee_count > 0
ORDER BY year_month, company_id, department_id;



#KPI 8: Total Cost per Department

In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_cost_per_department AS
SELECT 
  year_month,
  year,
  month,
  company_id,
  company_name,
  department_id,
  department_name,
  ROUND(payroll_total_compensation, 2) AS payroll_cost,
  ROUND(gl_total_expenses, 2) AS other_expenses,
  ROUND(total_departmental_cost, 2) AS total_department_cost,
  current_timestamp() AS last_updated
FROM mini_project.gold.datacube_kpi
ORDER BY year_month, company_id, total_department_cost DESC;



#KPI 9: Headcount Distribution by Department


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_headcount_by_department AS
SELECT 
  e.company_id,
  c.company_name,
  e.department_id,
  d.department_name,
  COUNT(DISTINCT e.employee_id) AS active_headcount,
  COUNT(DISTINCT CASE WHEN e.position LIKE '%Manager%' OR e.position LIKE '%Director%' OR e.position LIKE '%VP%' OR e.position LIKE '%Chief%' THEN e.employee_id END) AS management_count,
  COUNT(DISTINCT CASE WHEN NOT (e.position LIKE '%Manager%' OR e.position LIKE '%Director%' OR e.position LIKE '%VP%' OR e.position LIKE '%Chief%') THEN e.employee_id END) AS individual_contributor_count,
  ROUND(AVG(e.base_salary), 2) AS avg_base_salary,
  current_timestamp() AS last_updated
FROM mini_project.gold.dim_employee e
INNER JOIN mini_project.gold.dim_company c ON e.company_id = c.company_id
INNER JOIN mini_project.gold.dim_department d ON e.department_id = d.department_id
WHERE e.is_currently_active = TRUE
GROUP BY e.company_id, c.company_name, e.department_id, d.department_name
ORDER BY company_id, active_headcount DESC;



#KPI 10: Payroll Cost as % of Company Revenue (Labor Efficiency Ratio)


In [0]:
CREATE OR REPLACE TABLE mini_project.gold.kpi_payroll_revenue_ratio AS
SELECT 
  year_month,
  year,
  month,
  company_id,
  company_name,
  ROUND(SUM(payroll_total_compensation), 2) AS total_payroll_cost,
  ROUND(SUM(gl_revenue), 2) AS total_revenue,
  CASE 
    WHEN SUM(gl_revenue) = 0 THEN NULL
    ELSE ROUND((SUM(payroll_total_compensation) / SUM(gl_revenue)) * 100, 2)
  END AS payroll_as_pct_of_revenue,
  CASE 
    WHEN SUM(payroll_total_compensation) = 0 THEN NULL
    ELSE ROUND(SUM(gl_revenue) / SUM(payroll_total_compensation), 2)
  END AS revenue_per_payroll_dollar,
  current_timestamp() AS last_updated
FROM mini_project.gold.datacube_kpi
GROUP BY year_month, year, month, company_id, company_name
ORDER BY year_month, company_id;

